In [0]:
from pyspark.sql import functions as F, DataFrame
from datetime import datetime

# ── Configuração ──────────────────────────────────────────────────────────────
SOURCE_TABLE = "homologacao.salutar_saude.raw_operadoras"
TARGET_TABLE = "homologacao.salutar_saude.silver_operadoras"

DATE_COLS    = []                    # sem colunas de data nesta tabela
INITCAP_COLS = []                    # sem campos categóricos nesta tabela
TRIM_COLS    = ["operadora_nome"]    # trim: remove espaços extras do nome
MERGE_KEY    = "operadora_id"        # chave primária para o MERGE incremental

run_ts = datetime.now()

print(f"Pipeline  : silver_operadoras")
print(f"Iniciado  : {run_ts:%Y-%m-%d %H:%M:%S}")
print(f"Origem    : {SOURCE_TABLE}")
print(f"Destino   : {TARGET_TABLE}")

Pipeline  : silver_operadoras
Iniciado  : 2026-07-04 19:01:13
Origem    : homologacao.salutar_saude.raw_operadoras
Destino   : homologacao.salutar_saude.silver_operadoras


In [0]:
df_raw = spark.table(SOURCE_TABLE)
n_raw = df_raw.count()

print(f"[OK] {n_raw:,} registros lidos de {SOURCE_TABLE}")

[OK] 8 registros lidos de homologacao.salutar_saude.raw_operadoras


In [0]:
def parse_date(col_name: str) -> F.Column:
    """Normaliza datas para YYYY-MM-DD.
    Suporta os formatos: YYYY-MM-DD, DD-MM-YYYY, DD/MM/YYYY.
    Usa try_to_date para retornar NULL em vez de lançar exceção.
    """
    return F.date_format(
        F.coalesce(
            F.expr(f"try_to_date(`{col_name}`, 'yyyy-MM-dd')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd-MM-yyyy')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd/MM/yyyy')"),
        ),
        "yyyy-MM-dd",
    ).alias(col_name)


def parse_initcap(col_name: str) -> F.Column:
    """Normaliza campos categóricos: remove espaços extras e padroniza capitalização.
    Não usar em siglas/códigos (ex.: UF). Ex.: 'GRANDE' → 'Grande' | 'média ' → 'Média'.
    """
    return F.initcap(F.trim(F.col(col_name))).alias(col_name)


def parse_trim(col_name: str) -> F.Column:
    """Remove espaços extras de campos de texto livres (nome, descrição)."""
    return F.trim(F.col(col_name)).alias(col_name)


def transform(df: DataFrame, cols: list) -> DataFrame:
    """Aplica todas as transformações de padronização da camada silver.
    Remove a coluna _rescued_data (artefato da camada RAW) e
    adiciona _updated_at com o timestamp de execução.

    Args:
        df  : DataFrame de origem (camada RAW).
        cols: lista de colunas pré-computada fora da função (evita RPC repetido
              ao acessar df.columns dentro de um loop/função no Spark Connect).
    """
    return df.select(
        *[
            parse_date(c)    if c in DATE_COLS    else
            parse_initcap(c) if c in INITCAP_COLS else
            parse_trim(c)    if c in TRIM_COLS    else
            F.col(c)
            for c in cols
        ],
        F.lit(run_ts).cast("timestamp").alias("_updated_at"),  # em select, não withColumn
    )


print("[OK] Funções de transformação definidas.")

[OK] Funções de transformação definidas.


In [0]:
# Computa schema uma única vez fora da função (evita RPC repetido no Spark Connect)
silver_cols = [c for c in df_raw.columns if c != "_rescued_data"]
df_silver = transform(df_raw, silver_cols)

# ── Validação de qualidade ────────────────────────────────────────────────────
DQ_COLS = DATE_COLS + INITCAP_COLS + TRIM_COLS  # colunas a verificar nulos

null_counts = (
    df_silver
    .select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in DQ_COLS])
    .collect()[0]
    .asDict()
) if DQ_COLS else {}

n_silver = df_silver.count()

# Duplicatas de chave na fonte (serão removidas no passo de escrita)
dup_keys_df  = df_silver.groupBy(MERGE_KEY).count().filter(F.col("count") > 1)
n_dup_keys   = dup_keys_df.count()
dup_ids_list = [str(r[MERGE_KEY]) for r in dup_keys_df.collect()] if n_dup_keys > 0 else []

print("── Data Quality ───────────────────────────────────────────────────────────")
print(f"  Total de registros           : {n_silver:,}")
print(f"  Correspondência com RAW      : {'[OK]' if n_silver == n_raw else '[WARN] divergência!'}")
for col_name, nulls in null_counts.items():
    tag = "[WARN]" if nulls > 0 else "[OK]  "
    print(f"  {tag} {col_name}: {nulls:,} nulos")
tag = "[WARN]" if n_dup_keys > 0 else "[OK]  "
detail = f" → ids: {dup_ids_list}" if dup_ids_list else ""
print(f"  {tag} {MERGE_KEY} duplicados na fonte           : {n_dup_keys:,}{detail}")
print("─" * 65)

assert n_silver == n_raw, f"Contagem divergente: RAW={n_raw}, silver={n_silver}"

── Data Quality ───────────────────────────────────────────────────────────
  Total de registros           : 8
  Correspondência com RAW      : [OK]
  [OK]   operadora_nome: 0 nulos
  [OK]   operadora_id duplicados na fonte           : 0
─────────────────────────────────────────────────────────────────


In [0]:
from delta.tables import DeltaTable

# ── Escrita incremental via MERGE ─────────────────────────────────────────────
# Estratégia de deduplicacão — 2 etapas:
# 1. dropDuplicates()            → remove linhas 100% idênticas
# 2. dropDuplicates([MERGE_KEY]) → garante cardinalidade 1:1 por chave para o MERGE Delta
df_to_merge = df_silver.dropDuplicates().dropDuplicates([MERGE_KEY])

if spark.catalog.tableExists(TARGET_TABLE):
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)

    (
        delta_target.alias("target")
        .merge(
            df_to_merge.alias("source"),
            f"target.{MERGE_KEY} = source.{MERGE_KEY}"
        )
        .whenMatchedUpdateAll()       # atualiza operadora existente se houver mudança
        .whenNotMatchedInsertAll()    # insere nova operadora
        # .whenNotMatchedBySourceDelete()  # descomente para remover operadoras deletadas da RAW
        .execute()
    )
    print(f"[OK] MERGE concluído      : {TARGET_TABLE}")
else:
    # Primeira execução: cria a tabela (carga inicial)
    df_to_merge.write.format("delta").saveAsTable(TARGET_TABLE)
    print(f"[OK] Carga inicial        : {TARGET_TABLE}")

n_written = spark.table(TARGET_TABLE).count()
print(f"[OK] Registros na tabela  : {n_written:,}")
print(f"[OK] Fim do pipeline      : {datetime.now():%Y-%m-%d %H:%M:%S}")

[OK] Carga inicial        : homologacao.salutar_saude.silver_operadoras
[OK] Registros na tabela  : 8
[OK] Fim do pipeline      : 2026-07-04 19:01:29


In [0]:
%sql
SELECT * FROM homologacao.salutar_saude.silver_operadoras
ORDER BY operadora_id

operadora_id,operadora_nome,_updated_at
1,Vitamed,2026-07-04T19:01:13.268Z
2,Boavida Saúde,2026-07-04T19:01:13.268Z
3,SulVida Seguros,2026-07-04T19:01:13.268Z
4,NovaClin Intermédica,2026-07-04T19:01:13.268Z
5,Sanare Saúde,2026-07-04T19:01:13.268Z
6,Aurora Saúde,2026-07-04T19:01:13.268Z
7,Unicare Coop,2026-07-04T19:01:13.268Z
8,PlenaCare,2026-07-04T19:01:13.268Z
